# Segmentação de clientes → personas (K-Means)

Agrupa clientes por **quem são** (perfil) e **como compram e engajam**
(comportamento) para construir personas de negócio, e lê a resposta a oferta
dentro de cada uma. A modelagem é a **nível cliente** (uma linha por
`account_id`) e **descritiva**: usa a janela inteira do teste, então nomeia
segmentos e explica resultados — nunca vira feature do X-learner (isso seria
leakage, G2).

Toda a lógica vive em `src/clustering/`; aqui só importamos, executamos e lemos.
Spark carrega e reduz ao grão cliente; pandas faz a análise.

Seções: 1. Setup & carga · 2. Grão cliente · 3. Tratamento das variáveis ·
4. Escolha de k · 5. Ajuste · 6. Perfil dos clusters · 7. Personas ·
8. Resposta por persona · 9. Síntese.

## 1. Setup & carga

Spark carrega os eventos brutos e o grão processado; o tema executivo de `src/viz.py` é aplicado uma vez.

In [1]:
import os, sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "config.yaml").exists():
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd

from src import clustering, viz
from src.config import load
from src.io import parse_events
from src.pipeline import build_spark

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)
viz.setup_theme()

cfg = load()
spark = build_spark(cfg, app_name="clustering")
spark.sparkContext.setLogLevel("ERROR")

events = parse_events(spark, cfg).cache()
processed = spark.read.parquet(str(cfg.processed_dir)).cache()
print(f"eventos: {events.count():,} | linhas processadas: {processed.count():,}")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/07/12 15:30:50 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


eventos: 306,534 | linhas processadas: 76,277


## 2. Grão cliente

Uma linha por `account_id`: perfil, compras (janela inteira), engajamento e resposta observada. Só as features de `cfg.cluster_features` entram no ajuste; `conv_rate`/`margem` ficam para a leitura.

In [2]:
clientes = clustering.build_client_frame(processed, events)
print(f"{len(clientes):,} clientes | {int(clientes['identity_missing'].sum()):,} com identidade ausente")
print(f"features de ajuste: {cfg.cluster_features}")
clientes.head()

16,994 clientes | 2,174 com identidade ausente
features de ajuste: ['age', 'credit_card_limit', 'tenure_days', 'spend_total', 'txn_count', 'avg_ticket', 'view_rate']


,account_id,age,credit_card_limit,tenure_days,identity_missing,gender,offers_received,offers_viewed,conversions,conversion_value,reward_cost,spend_total,txn_count,avg_ticket,view_rate,conv_rate,margem
0,0009655768c64bdeb2e877511632db8f,33.0,72000.0,461,0,M,5,4,4,58.40,7.0,127.60,8,15.950000,0.8,0.8,51.40
1,00116118485d4dfda04fdbaba9a87b5c,NaN,NaN,92,1,unknown,2,2,0,0.00,0.0,4.09,3,1.363333,1.0,0.0,0.00
2,0011e0d4e6b944f998e987f904e8c1e5,40.0,57000.0,198,0,O,5,5,3,42.94,13.0,79.46,5,15.892000,1.0,0.6,29.94
3,0020c2b971eb4e9188eac86d93036a77,59.0,90000.0,874,0,F,5,2,2,34.87,4.0,196.86,8,24.607500,0.4,0.4,30.87
4,0020ccbbb6d84e358d3414a3ff76cffd,24.0,60000.0,622,0,F,4,4,4,55.60,13.0,154.05,12,12.837500,1.0,1.0,42.60


In [3]:
# Perfil univariado das features de ajuste, em unidade original — as caudas
# monetárias e o segmento sentinela (sem age/limite) que motivam o tratamento.
resumo = clientes[list(cfg.cluster_features)].describe(percentiles=[.25, .5, .75, .99]).T
resumo["nulos"] = clientes[list(cfg.cluster_features)].isna().sum().values
resumo.round(2)

,count,mean,std,min,25%,50%,75%,99%,max,nulos
age,14820.0,54.39,17.38,18.0,42.00,55.00,66.00,92.00,101.00,2174
credit_card_limit,14820.0,65406.88,21598.06,30000.0,49000.00,64000.00,80000.00,117000.00,120000.00,2174
tenure_days,16994.0,517.44,411.27,0.0,208.00,358.00,791.00,1737.00,1823.00,0
spend_total,16994.0,104.46,125.94,0.0,21.82,69.41,148.83,693.96,1608.69,0
txn_count,16994.0,8.18,5.12,0.0,4.00,7.00,11.00,23.00,36.00,0
avg_ticket,16994.0,13.34,16.00,0.0,3.04,11.41,20.24,68.19,451.47,0
view_rate,16994.0,0.72,0.24,0.0,0.50,0.75,1.00,1.00,1.00,0


## 3. Tratamento das variáveis

Três decisões fazem a distância euclidiana significar algo (todas em `clustering.design_matrix`): **sem imputação** do segmento `identity_missing` (ele já é um segmento, fica fora do ajuste), **`log1p`** nas caudas monetárias (`spend_total`/`txn_count`/`avg_ticket`), e **z-score** depois (features em unidades distintas).

In [4]:
dm = clustering.design_matrix(clientes, cfg)
print(f"matriz de ajuste: {dm.X.shape[0]:,} clientes × {dm.X.shape[1]} features")
print(f"log1p aplicado em: {dm.log_columns}")
print(f"padronização — média máx |{np.abs(dm.X.mean(axis=0)).max():.2e}|, "
      f"desvio máx |{np.abs(dm.X.std(axis=0) - 1).max():.2e}| de 1")

matriz de ajuste: 14,820 clientes × 7 features
log1p aplicado em: ['spend_total', 'txn_count', 'avg_ticket']
padronização — média máx |8.05e-17|, desvio máx |2.22e-16| de 1


In [5]:
# spend_total antes/depois do log1p: a cauda que dominaria a distância euclidiana.
completos = clientes.loc[clientes["identity_missing"] == 0]
h_bruto = np.histogram(completos["spend_total"], bins=cfg.histogram_bins)
h_log = np.histogram(np.log1p(completos["spend_total"]), bins=cfg.histogram_bins)
paineis = [
    {"title": "spend_total (bruto): cauda longa",
     "x": ((h_bruto[1][:-1] + h_bruto[1][1:]) / 2).tolist(), "y": h_bruto[0].tolist()},
    {"title": "log1p(spend_total): simétrico",
     "x": ((h_log[1][:-1] + h_log[1][1:]) / 2).tolist(), "y": h_log[0].tolist()},
]
viz.faceted(
    paineis, kind="bar", cols=2, row_height=340,
    title="log1p tira a cauda que ditaria os centróides",
    subtitle=f"n={len(completos):,} clientes de perfil completo",
).show()

## 4. Escolha de k

Varredura de `k` com quatro índices na mesma matriz padronizada: inércia (cotovelo), silhouette (critério de decisão, ↑), Davies-Bouldin (↓) e Calinski-Harabasz (↑). O critério fica visível — nunca só o vencedor.

In [6]:
scan = clustering.scan_k(dm.X, cfg)
k = clustering.choose_k(scan)
print(f"k escolhido (silhouette máx): {k}")
scan.round(4)

k escolhido (silhouette máx): 2


,k,inercia,silhouette,davies_bouldin,calinski_harabasz
0,2,77887.1938,0.2483,1.6304,4918.5057
1,3,63883.6225,0.2229,1.5458,4622.1042
2,4,57931.1184,0.1959,1.5986,3905.2536
3,5,52483.6825,0.2057,1.4571,3617.1515
4,6,47857.1149,0.2069,1.4176,3459.6804
5,7,43770.1096,0.2085,1.3772,3382.5750
6,8,41090.2237,0.1996,1.4185,3226.2496


In [7]:
# Os quatro índices lado a lado; a linha marca o k escolhido. Grandezas sem
# unidade comum nunca vão em eixo y duplo — cada painel tem a sua escala.
melhor_sil = scan.loc[scan["silhouette"].idxmax(), "silhouette"]
faixa_sil = scan["silhouette"].max() - scan["silhouette"].min()
ks = scan["k"].tolist()
viz.faceted(
    [
        {"title": "inércia (cotovelo, ↓)", "x": ks, "y": scan["inercia"].tolist()},
        {"title": "silhouette (critério, ↑)", "x": ks, "y": scan["silhouette"].tolist()},
        {"title": "Davies-Bouldin (↓)", "x": ks, "y": scan["davies_bouldin"].tolist()},
        {"title": "Calinski-Harabasz (↑)", "x": ks, "y": scan["calinski_harabasz"].tolist()},
    ],
    kind="line", cols=2, vline=k, xlabel="k (número de clusters)", row_height=320,
    title=f"k = {k} maximiza a silhouette ({melhor_sil:.3f}); a curva é rasa (amplitude {faixa_sil:.3f})",
    subtitle="segmentos são cortes num gradiente de valor, não espécies distintas",
).show()

## 5. Ajuste

K-Means com o k escolhido, determinístico por `cfg.seed`. O segmento de identidade ausente, fora do ajuste, é re-anexado como segmento próprio (não imputado).

In [8]:
rotulos = clustering.fit(dm.X, dm.account_ids, k, cfg)
segmentos = clustering.assign_segments(clientes, rotulos)
segmentos.groupby("segmento").size().rename("clientes").reset_index()

,segmento,clientes
0,cluster 0,8706
1,cluster 1,6114
2,identidade ausente,2174


## 6. Perfil dos clusters

Média de cada feature por segmento, em unidade original. O heatmap mostra o z-score entre segmentos (a média bruta não é comparável entre reais/anos/dias); o valor bruto vai anotado na célula.

In [9]:
perfil = clustering.profile_segments(segmentos)
perfil.round(2)

,segmento,clientes,fracao,age,credit_card_limit,tenure_days,spend_total,txn_count,avg_ticket,offers_received,view_rate,conv_rate,margem
0,cluster 0,8706,0.51,59.58,76154.95,564.20,172.24,8.29,22.31,4.49,0.77,0.65,55.39
1,cluster 1,6114,0.36,47.00,50102.22,463.03,38.47,8.47,4.37,4.48,0.63,0.27,6.63
2,identidade ausente,2174,0.13,NaN,NaN,483.23,18.63,6.90,2.65,4.50,0.76,0.16,3.91


In [10]:
# z-score por coluna entre segmentos; valor bruto sobreposto. view_rate/conv_rate
# ficaram fora do ajuste (são a resposta), mostrados aqui só para leitura.
colunas_perfil = [c for c in list(cfg.cluster_features) + ["conv_rate"] if c in perfil.columns]
valores = perfil.set_index("segmento")[colunas_perfil].astype("float64")
z = (valores - valores.mean()) / valores.std(ddof=0).replace(0, np.nan)
viz.heatmap(
    z, diverging=True, annotate=True, text=valores.to_numpy(), text_template="%{text:.4g}",
    colorbar_title="z", reverse_y=False,
    title="Cada segmento é um ponto no gradiente de valor e engajamento",
    subtitle="z-score entre segmentos · valor bruto anotado",
).show()

## 7. Personas

Os rótulos do K-Means (`cluster 0/1/…`) são arbitrários; `name_personas` os traduz para rótulos de negócio derivados do próprio perfil (ordem por valor econômico), mais o segmento sem cadastro.

In [11]:
personas = clustering.name_personas(perfil)
cartao = personas[["persona", "segmento", "clientes", "fracao"] +
                  [c for c in ["spend_total", "avg_ticket", "txn_count", "tenure_days",
                               "credit_card_limit", "age", "view_rate", "conv_rate"]
                   if c in personas.columns]]
cartao.sort_values("clientes", ascending=False).round(2).reset_index(drop=True)

,persona,segmento,clientes,fracao,spend_total,avg_ticket,txn_count,tenure_days,credit_card_limit,age,view_rate,conv_rate
0,Alto valor,cluster 0,8706,0.51,172.24,22.31,8.29,564.20,76154.95,59.58,0.77,0.65
1,Baixo ticket,cluster 1,6114,0.36,38.47,4.37,8.47,463.03,50102.22,47.00,0.63,0.27
2,Sem cadastro completo,identidade ausente,2174,0.13,18.63,2.65,6.90,483.23,NaN,NaN,0.76,0.16


## 8. Resposta por persona

Como cada persona responde a cada tipo de oferta — a leitura de negócio que fecha a segmentação. Observacional (média do que aconteceu), não o efeito causal de enviar; denominadores explícitos.

In [12]:
nome_por_segmento = personas.set_index("segmento")["persona"].to_dict()
resposta = clustering.segment_response(
    processed, spark.createDataFrame(segmentos[["account_id", "segmento"]]))
resposta["persona"] = resposta["segmento"].map(nome_por_segmento)
resposta.round(4)

,segmento,offer_type,envios,vistos,conversoes,receita,custo,taxa_view,taxa_conversao,taxa_conversao_vistos,margem_por_envio,persona
0,cluster 0,bogo,15738,13101,10585,240947.75,78900.0,0.8324,0.6726,0.8080,10.2966,Alto valor
1,cluster 0,discount,15504,11353,10776,277816.23,30347.0,0.7323,0.6950,0.9492,15.9616,Alto valor
2,cluster 0,informational,7864,5460,3684,72749.88,0.0,0.6943,0.4685,0.6747,9.2510,Alto valor
3,cluster 1,bogo,10799,8163,2547,25569.11,15885.0,0.7559,0.2359,0.3120,0.8968,Baixo ticket
4,cluster 1,discount,11160,6088,1584,21439.71,4058.0,0.5455,0.1419,0.2602,1.5575,Baixo ticket
5,cluster 1,informational,5436,3067,3260,13498.53,0.0,0.5642,0.5997,1.0629,2.4832,Baixo ticket
6,identidade ausente,bogo,3962,3250,364,3920.58,2010.0,0.8203,0.0919,0.1120,0.4822,Sem cadastro completo
7,identidade ausente,discount,3879,2821,140,4133.13,368.0,0.7272,0.0361,0.0496,0.9706,Sem cadastro completo
8,identidade ausente,informational,1935,1351,1064,2821.58,0.0,0.6982,0.5499,0.7876,1.4582,Sem cadastro completo


In [13]:
# Margem por envio (receita − custo) / envios, uma barra por tipo de oferta,
# agrupadas por persona — onde uma política uniforme paga a conta.
melhor = resposta.loc[resposta["margem_por_envio"].idxmax()]
pior = resposta.loc[resposta["margem_por_envio"].idxmin()]
tipos = sorted(resposta["offer_type"].unique())
margem_wide = (
    resposta.pivot(index="persona", columns="offer_type", values="margem_por_envio")
    .reset_index()
)
viz.grouped_bars(
    margem_wide, category="persona", series=tipos, value_label="%{y:.1f}",
    ylabel="margem por envio (R$)",
    title=f"Margem por envio vai de {pior['margem_por_envio']:.1f} ({pior['persona']}, {pior['offer_type']}) "
          f"a {melhor['margem_por_envio']:.1f} ({melhor['persona']}, {melhor['offer_type']})",
    subtitle="média observacional por envio · não é o efeito causal de enviar",
).show()

## 9. Síntese

As personas e o que cada uma diz para a decisão de oferta. Descritivo — o efeito causal de enviar é o que a spec 02 estima; aqui fixamos quem é quem e onde a resposta observada varia.

In [14]:
sintese = personas[["persona", "clientes", "fracao"]].copy()
sintese = sintese.merge(
    resposta.groupby("persona").agg(
        margem_media=("margem_por_envio", "mean"),
        melhor_oferta=("offer_type", lambda s: resposta.loc[s.index].sort_values(
            "margem_por_envio").iloc[-1]["offer_type"]),
    ).reset_index(),
    on="persona", how="left",
)
sintese.sort_values("clientes", ascending=False).round(3).reset_index(drop=True)

,persona,clientes,fracao,margem_media,melhor_oferta
0,Alto valor,8706,0.512,11.836,discount
1,Baixo ticket,6114,0.360,1.646,informational
2,Sem cadastro completo,2174,0.128,0.970,informational
